# Notebook 13 — PyTorch Tensors

## Leadership and Management Book Recommendation System

### Objective

This notebook prepares the numerical representations developed in the previous stages for use with **PyTorch**.

The project is primarily an **unsupervised, content-based recommendation system**. Therefore, this notebook does not create an artificial target variable or convert the project into a supervised learning problem.

Instead, the notebook focuses on converting validated numerical book representations into PyTorch tensors while preserving alignment with the existing book catalog.

### Inputs

The notebook uses outputs generated and validated in previous stages, including:

- Book identifiers and metadata
- TF-IDF representations from NLP processing
- Reduced-dimensional representations produced using Truncated SVD / Latent Semantic Analysis (LSA)
- Topic cluster information where appropriate

### Workflow

1. Validate the Python and PyTorch environment
2. Load the required processed artifacts
3. Verify book-level alignment
4. Select the appropriate numerical representation
5. Convert NumPy / sparse representations into PyTorch tensors
6. Inspect tensor shapes, dimensions, and data types
7. Demonstrate basic tensor operations relevant to the recommendation system
8. Validate tensor integrity
9. Save reusable tensor artifacts if required

### Methodological Note

PyTorch is used here as a numerical and deep-learning framework rather than as evidence that the recommendation problem is supervised.

No synthetic target variable is introduced.

The existing recommendation system remains based on:

**Enriched TF-IDF → Cosine Similarity → Duplicate Suppression → Ranked Top-N Recommendations**

PyTorch tensors provide a compatible numerical representation for subsequent neural-network experimentation and application development.

In [3]:
# ============================================================
# NOTEBOOK 13 — PYTORCH TENSORS
# ENVIRONMENT AND IMPORT VALIDATION
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    import torch
except ImportError:
    torch = None


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/Users/jannoelvero/Documents/Ironhack/Week_11/"
    "Leadership_Management_Book_Recommendation_System"
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"


# ------------------------------------------------------------
# Environment validation
# ------------------------------------------------------------

print("PYTORCH TENSOR NOTEBOOK — ENVIRONMENT CHECK")
print("=" * 80)

print("Python version:")
print(sys.version)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nProject root exists:")
print(PROJECT_ROOT.exists())

print("\nPyTorch installed:")
print(torch is not None)

if torch is not None:
    print("\nPyTorch version:")
    print(torch.__version__)

    print("\nMPS available:")
    print(
        torch.backends.mps.is_available()
        if hasattr(torch.backends, "mps")
        else False
    )

PYTORCH TENSOR NOTEBOOK — ENVIRONMENT CHECK
Python version:
3.13.11 | packaged by Anaconda, Inc. | (main, Dec 10 2025, 21:21:08) [Clang 20.1.8 ]

Project root:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System

Project root exists:
True

PyTorch installed:
True

PyTorch version:
2.14.0

MPS available:
True


## 1. Load the Numerical Book Representation

The recommendation system uses sparse enriched TF-IDF vectors for exact cosine-similarity recommendations.

For PyTorch tensor preparation, this notebook uses the **200-dimensional enriched LSA representation** produced using Truncated SVD in Notebook 10.

This representation was selected because:

- it provides a compact dense numerical representation of book content;
- it preserves information from the enriched TF-IDF feature space;
- it is directly compatible with PyTorch tensors;
- it avoids unnecessarily converting the original sparse TF-IDF matrix into a large dense tensor.

The enriched LSA representation contains **200 numerical dimensions per book** and remains aligned with the NLP book index.

The original enriched TF-IDF representation remains the production representation for the content-based recommendation engine.

In [4]:
# ============================================================
# LOCATE ENRICHED LSA / SVD ARTIFACTS
# ============================================================

print("AVAILABLE LSA / SVD ARTIFACTS")
print("=" * 80)

candidate_files = []

for directory in [
    MODELS_DIR,
    PROCESSED_DIR
]:
    for path in directory.glob("*"):
        name = path.name.lower()

        if (
            "lsa" in name
            or "svd" in name
        ):
            candidate_files.append(path)


for path in sorted(candidate_files):
    print(path)

AVAILABLE LSA / SVD ARTIFACTS
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/core_lsa_200.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/core_lsa_topic_terms.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/enriched_lsa_200.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/enriched_lsa_topic_terms.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/svd_component_search.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/svd_neighbor_preservation.csv
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/svd_similarity_preservation.csv
/Users/jannoelvero/Do

### Selected Tensor Input

The PyTorch tensor is created from:

`data/processed/enriched_lsa_200.csv`

This artifact contains the transformed 200-dimensional enriched LSA representation generated in the dimensionality-reduction stage.

The fitted `enriched_svd_200.joblib` model is not used directly as the tensor input because it represents the transformation model rather than the transformed book-level feature matrix.

Before tensor conversion, the dataset is validated for:

- number of books;
- number of LSA dimensions;
- book identifier uniqueness;
- missing values;
- numerical feature types;
- alignment with the NLP book index.

In [5]:
# ============================================================
# LOAD AND INSPECT ENRICHED LSA REPRESENTATION
# ============================================================

ENRICHED_LSA_PATH = (
    PROCESSED_DIR
    / "enriched_lsa_200.csv"
)

enriched_lsa = pd.read_csv(
    ENRICHED_LSA_PATH
)


print("ENRICHED LSA REPRESENTATION")
print("=" * 80)

print("Path:")
print(ENRICHED_LSA_PATH)

print("\nShape:")
print(enriched_lsa.shape)

print("\nFirst 10 columns:")
print(
    enriched_lsa.columns[:10].tolist()
)

print("\nLast 10 columns:")
print(
    enriched_lsa.columns[-10:].tolist()
)

print("\nData types:")
print(
    enriched_lsa.dtypes.value_counts()
)

print("\nMissing values:")
print(
    enriched_lsa.isna().sum().sum()
)

print("\nFirst 3 rows:")
display(
    enriched_lsa.head(3)
)

ENRICHED LSA REPRESENTATION
Path:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/enriched_lsa_200.csv

Shape:
(2067, 202)

First 10 columns:
['book_id', 'is_zero_vector', 'LSA_1', 'LSA_2', 'LSA_3', 'LSA_4', 'LSA_5', 'LSA_6', 'LSA_7', 'LSA_8']

Last 10 columns:
['LSA_191', 'LSA_192', 'LSA_193', 'LSA_194', 'LSA_195', 'LSA_196', 'LSA_197', 'LSA_198', 'LSA_199', 'LSA_200']

Data types:
float64    200
str          1
bool         1
Name: count, dtype: int64

Missing values:
0

First 3 rows:


,book_id,is_zero_vector,LSA_1,LSA_2,LSA_3,LSA_4,LSA_5,LSA_6,LSA_7,LSA_8,...,LSA_191,LSA_192,LSA_193,LSA_194,LSA_195,LSA_196,LSA_197,LSA_198,LSA_199,LSA_200
0,BOOK00001,False,0.107705,0.074201,0.009673,0.046081,0.048191,-0.018987,0.019303,0.011081,...,-0.021581,-0.055354,-0.012586,0.016467,0.009915,-0.035356,-0.009720,-0.024113,-0.043064,-0.058543
1,BOOK00002,False,0.150275,0.106579,-0.018099,0.056580,-0.055400,0.030764,-0.024646,-0.024983,...,0.012678,0.009395,0.038798,-0.023484,0.013635,-0.012947,0.030716,0.015020,0.001142,-0.012212
2,BOOK00003,False,0.285356,0.771247,0.133419,-0.096192,0.045057,-0.023330,-0.014919,-0.053478,...,-0.013281,-0.004609,0.003108,-0.013560,0.004811,-0.000718,-0.008357,0.000862,-0.001921,-0.009921


## 2. Validate LSA Features and Book Alignment

The enriched LSA dataset contains 202 columns:

- `book_id` — book identifier
- `is_zero_vector` — indicator inherited from the enriched TF-IDF representation
- `LSA_1` to `LSA_200` — numerical latent semantic features

Only the **200 LSA dimensions** will be converted into the primary PyTorch feature tensor.

`book_id` is retained separately to preserve book-to-tensor-row alignment, while `is_zero_vector` is retained as validation metadata rather than used as a semantic feature.

Before tensor conversion, the following checks are performed:

- exactly 200 LSA feature columns are present;
- book identifiers are unique;
- LSA rows align with the NLP book index;
- zero-vector status is preserved;
- all selected features are numerical and complete.

In [6]:
# ============================================================
# VALIDATE LSA FEATURES AND BOOK ALIGNMENT
# ============================================================

# Identify the 200 LSA feature columns
lsa_columns = [
    col
    for col in enriched_lsa.columns
    if col.startswith("LSA_")
]


print("LSA FEATURE VALIDATION")
print("=" * 80)

print(
    "Number of LSA features:",
    len(lsa_columns)
)

print(
    "Expected 200 features:",
    len(lsa_columns) == 200
)

print(
    "\nBook IDs unique:",
    enriched_lsa["book_id"].is_unique
)

print(
    "Unique book IDs:",
    enriched_lsa["book_id"].nunique()
)

print(
    "\nZero-vector books:",
    enriched_lsa["is_zero_vector"].sum()
)

print(
    "Non-zero-vector books:",
    (~enriched_lsa["is_zero_vector"]).sum()
)


# ------------------------------------------------------------
# Validate against NLP index
# ------------------------------------------------------------

NLP_INDEX_PATH = (
    PROCESSED_DIR
    / "nlp_book_index.csv"
)

nlp_index = pd.read_csv(
    NLP_INDEX_PATH
)

print(
    "\nNLP index rows:",
    len(nlp_index)
)

print(
    "LSA rows:",
    len(enriched_lsa)
)

print(
    "Same number of books:",
    len(nlp_index) == len(enriched_lsa)
)

print(
    "Exact book ID order aligned:",
    enriched_lsa["book_id"]
    .reset_index(drop=True)
    .equals(
        nlp_index["book_id"]
        .reset_index(drop=True)
    )
)


# ------------------------------------------------------------
# Validate numerical matrix
# ------------------------------------------------------------

lsa_matrix = enriched_lsa[
    lsa_columns
].to_numpy()

print(
    "\nNumPy matrix shape:",
    lsa_matrix.shape
)

print(
    "All values finite:",
    np.isfinite(lsa_matrix).all()
)

print(
    "Missing values:",
    np.isnan(lsa_matrix).sum()
)

LSA FEATURE VALIDATION
Number of LSA features: 200
Expected 200 features: True

Book IDs unique: True
Unique book IDs: 2067

Zero-vector books: 27
Non-zero-vector books: 2040

NLP index rows: 2067
LSA rows: 2067
Same number of books: True
Exact book ID order aligned: True

NumPy matrix shape: (2067, 200)
All values finite: True
Missing values: 0


## 3. Convert the LSA Representation to a PyTorch Tensor

The validated 200-dimensional LSA matrix is converted from NumPy into a PyTorch tensor.

The numerical values are converted from `float64` to **`float32`**, which is the standard precision used for most neural-network operations in PyTorch. This reduces memory requirements and provides compatibility with common PyTorch layers and hardware acceleration.

The resulting tensor has the structure:

**Books × LSA Features**

Each row represents one book and each column represents one latent semantic dimension.

The tensor remains aligned with the original `book_id` ordering so that every tensor row can be mapped back to its corresponding book.

In [7]:
# ============================================================
# CREATE PYTORCH FEATURE TENSOR
# ============================================================

# Convert NumPy matrix to float32
lsa_matrix_float32 = lsa_matrix.astype(
    np.float32
)

# Create PyTorch tensor
book_feature_tensor = torch.from_numpy(
    lsa_matrix_float32
)


print("PYTORCH FEATURE TENSOR")
print("=" * 80)

print(
    "Tensor shape:",
    book_feature_tensor.shape
)

print(
    "Tensor dimensions:",
    book_feature_tensor.ndim
)

print(
    "Tensor dtype:",
    book_feature_tensor.dtype
)

print(
    "Tensor device:",
    book_feature_tensor.device
)

print(
    "Number of books:",
    book_feature_tensor.shape[0]
)

print(
    "Features per book:",
    book_feature_tensor.shape[1]
)

print(
    "\nContains NaN:",
    torch.isnan(
        book_feature_tensor
    ).any().item()
)

print(
    "Contains Inf:",
    torch.isinf(
        book_feature_tensor
    ).any().item()
)

print(
    "\nFirst book ID:",
    enriched_lsa.iloc[0]["book_id"]
)

print(
    "First tensor vector shape:",
    book_feature_tensor[0].shape
)

print(
    "\nFirst 10 tensor values:"
)

print(
    book_feature_tensor[0, :10]
)

PYTORCH FEATURE TENSOR
Tensor shape: torch.Size([2067, 200])
Tensor dimensions: 2
Tensor dtype: torch.float32
Tensor device: cpu
Number of books: 2067
Features per book: 200

Contains NaN: False
Contains Inf: False

First book ID: BOOK00001
First tensor vector shape: torch.Size([200])

First 10 tensor values:
tensor([ 0.1077,  0.0742,  0.0097,  0.0461,  0.0482, -0.0190,  0.0193,  0.0111,
         0.0039, -0.0295])


## 4. Tensor Operations for Book Representations

The PyTorch tensor can be used to perform numerical operations on the latent semantic representation of the books.

For this recommendation project, relevant tensor operations include:

- retrieving the numerical representation of an individual book;
- selecting multiple books as a batch;
- calculating vector norms;
- calculating cosine similarity between book vectors;
- transferring tensors between CPU and supported acceleration devices.

These operations demonstrate how the existing book representations can be used within PyTorch while maintaining their relationship with the content-based recommendation system.

The cosine similarities calculated from the LSA tensor are illustrative of operations in the reduced semantic space. They do not replace the production recommendation engine, which continues to use the original enriched TF-IDF representation.

In [8]:
# ============================================================
# BASIC PYTORCH OPERATIONS
# ============================================================

import torch.nn.functional as F


# ------------------------------------------------------------
# Select individual book vectors
# ------------------------------------------------------------

book_1 = book_feature_tensor[0]
book_2 = book_feature_tensor[1]


print("BASIC TENSOR OPERATIONS")
print("=" * 80)

print(
    "Book 1:",
    enriched_lsa.iloc[0]["book_id"]
)

print(
    "Book 2:",
    enriched_lsa.iloc[1]["book_id"]
)


# ------------------------------------------------------------
# Vector norms
# ------------------------------------------------------------

print(
    "\nBook 1 vector norm:",
    book_1.norm().item()
)

print(
    "Book 2 vector norm:",
    book_2.norm().item()
)


# ------------------------------------------------------------
# Cosine similarity
# ------------------------------------------------------------

pair_similarity = F.cosine_similarity(
    book_1.unsqueeze(0),
    book_2.unsqueeze(0)
)

print(
    "\nLSA cosine similarity:",
    pair_similarity.item()
)


# ------------------------------------------------------------
# Create a small batch
# ------------------------------------------------------------

book_batch = book_feature_tensor[:5]

print(
    "\nBatch shape:",
    book_batch.shape
)

print(
    "Batch dtype:",
    book_batch.dtype
)

print(
    "Batch device:",
    book_batch.device
)


# ------------------------------------------------------------
# Normalize vectors
# ------------------------------------------------------------

normalized_batch = F.normalize(
    book_batch,
    p=2,
    dim=1
)

normalized_norms = torch.linalg.vector_norm(
    normalized_batch,
    dim=1
)

print(
    "\nNormalized batch shape:",
    normalized_batch.shape
)

print(
    "Normalized vector norms:"
)

print(
    normalized_norms
)

BASIC TENSOR OPERATIONS
Book 1: BOOK00001
Book 2: BOOK00002

Book 1 vector norm: 0.5614725947380066
Book 2 vector norm: 0.5474666357040405

LSA cosine similarity: 0.045397255569696426

Batch shape: torch.Size([5, 200])
Batch dtype: torch.float32
Batch device: cpu

Normalized batch shape: torch.Size([5, 200])
Normalized vector norms:
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


## 5. Device Management and Apple Silicon Acceleration

PyTorch tensors can operate on different computing devices.

The canonical book feature tensor is retained on the **CPU** to keep the saved representation portable and independent of specific hardware.

Because this environment supports Apple's **Metal Performance Shaders (MPS)** backend, a copy of the tensor can be transferred to the Apple Silicon GPU when accelerated tensor computation is required.

This section verifies:

- automatic device selection;
- CPU-to-MPS tensor transfer;
- preservation of tensor shape and data type;
- successful numerical computation on the selected device.

The CPU tensor remains the canonical representation.

In [10]:
# ============================================================
# PYTORCH DEVICE MANAGEMENT
# ============================================================

# Select available acceleration device
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


print("PYTORCH DEVICE VALIDATION")
print("=" * 80)

print(
    "Selected device:",
    device
)

print(
    "Canonical tensor device:",
    book_feature_tensor.device
)


# ------------------------------------------------------------
# Transfer a COPY to selected device
# ------------------------------------------------------------

device_tensor = book_feature_tensor.to(
    device
)

print(
    "\nDevice tensor shape:",
    device_tensor.shape
)

print(
    "Device tensor dtype:",
    device_tensor.dtype
)

print(
    "Device tensor device:",
    device_tensor.device
)


# ------------------------------------------------------------
# Test computation on selected device
# ------------------------------------------------------------

device_norms = torch.linalg.vector_norm(
    device_tensor[:5],
    dim=1
)

print(
    "\nFirst 5 vector norms on device:"
)

print(
    device_norms
)


# ------------------------------------------------------------
# Confirm canonical tensor remains on CPU
# ------------------------------------------------------------

print(
    "\nCanonical tensor still on CPU:",
    book_feature_tensor.device.type == "cpu"
)

PYTORCH DEVICE VALIDATION
Selected device: mps
Canonical tensor device: cpu

Device tensor shape: torch.Size([2067, 200])
Device tensor dtype: torch.float32
Device tensor device: mps:0

First 5 vector norms on device:
tensor([0.5615, 0.5475, 0.9584, 0.5299, 0.4945], device='mps:0')

Canonical tensor still on CPU: True


## 6. PyTorch Dataset and DataLoader

PyTorch commonly organizes model-ready data using two components:

- **Dataset** — provides access to individual observations;
- **DataLoader** — retrieves observations in configurable batches.

For this project, each observation represents one book with:

- a book identifier;
- a 200-dimensional LSA feature vector.

No target variable is created because the current project remains an unsupervised/content-based recommendation problem.

The DataLoader prepares the feature vectors for efficient batch processing and provides the structure required for subsequent neural-network experimentation.

In [11]:
# ============================================================
# CREATE PYTORCH DATASET AND DATALOADER
# ============================================================

from torch.utils.data import Dataset, DataLoader


class BookFeatureDataset(Dataset):
    """
    PyTorch Dataset for book-level LSA features.
    """

    def __init__(
        self,
        book_ids,
        feature_tensor
    ):
        self.book_ids = list(book_ids)
        self.features = feature_tensor

    def __len__(self):
        return len(self.book_ids)

    def __getitem__(self, index):
        return {
            "book_id": self.book_ids[index],
            "features": self.features[index]
        }


# ------------------------------------------------------------
# Create Dataset
# ------------------------------------------------------------

book_dataset = BookFeatureDataset(
    book_ids=enriched_lsa["book_id"],
    feature_tensor=book_feature_tensor
)


# ------------------------------------------------------------
# Create DataLoader
# ------------------------------------------------------------

book_dataloader = DataLoader(
    book_dataset,
    batch_size=32,
    shuffle=False
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("PYTORCH DATASET AND DATALOADER")
print("=" * 80)

print(
    "Dataset size:",
    len(book_dataset)
)

print(
    "Batch size:",
    book_dataloader.batch_size
)

print(
    "Number of batches:",
    len(book_dataloader)
)


# Retrieve first batch
first_batch = next(
    iter(book_dataloader)
)

print(
    "\nFirst batch feature shape:",
    first_batch["features"].shape
)

print(
    "First batch dtype:",
    first_batch["features"].dtype
)

print(
    "First batch device:",
    first_batch["features"].device
)

print(
    "\nFirst 5 book IDs:"
)

print(
    first_batch["book_id"][:5]
)


# ------------------------------------------------------------
# Alignment check
# ------------------------------------------------------------

print(
    "\nFirst DataLoader ID matches source:",
    first_batch["book_id"][0]
    == enriched_lsa.iloc[0]["book_id"]
)

print(
    "First DataLoader vector matches tensor:",
    torch.equal(
        first_batch["features"][0],
        book_feature_tensor[0]
    )
)

PYTORCH DATASET AND DATALOADER
Dataset size: 2067
Batch size: 32
Number of batches: 65

First batch feature shape: torch.Size([32, 200])
First batch dtype: torch.float32
First batch device: cpu

First 5 book IDs:
['BOOK00001', 'BOOK00002', 'BOOK00003', 'BOOK00004', 'BOOK00005']

First DataLoader ID matches source: True
First DataLoader vector matches tensor: True


## 7. Zero-Vector Integrity Validation

The enriched NLP representation contains **27 books with zero TF-IDF vectors** because their available textual metadata did not produce usable features under the fitted vocabulary.

These observations were preserved during Truncated SVD and tensor conversion rather than artificially imputed or removed.

This section verifies that:

- the same 27 books remain zero vectors in the PyTorch representation;
- non-zero books retain numerical information;
- tensor conversion did not introduce or alter zero-vector observations.

Preserving these records maintains alignment with the complete 2,067-book catalog while allowing downstream processes to identify books without usable semantic representations.

In [12]:
# ============================================================
# ZERO-VECTOR INTEGRITY VALIDATION
# ============================================================

# Calculate L2 norm for every tensor row
tensor_norms = torch.linalg.vector_norm(
    book_feature_tensor,
    dim=1
)

# Identify zero vectors
tensor_zero_mask = torch.isclose(
    tensor_norms,
    torch.tensor(
        0.0,
        dtype=book_feature_tensor.dtype
    )
)

tensor_zero_count = int(
    tensor_zero_mask.sum().item()
)

source_zero_count = int(
    enriched_lsa["is_zero_vector"].sum()
)


print("ZERO-VECTOR INTEGRITY VALIDATION")
print("=" * 80)

print(
    "Source zero-vector count:",
    source_zero_count
)

print(
    "Tensor zero-vector count:",
    tensor_zero_count
)

print(
    "Counts match:",
    source_zero_count == tensor_zero_count
)


# ------------------------------------------------------------
# Compare exact zero-vector identities
# ------------------------------------------------------------

source_zero_ids = set(
    enriched_lsa.loc[
        enriched_lsa["is_zero_vector"],
        "book_id"
    ]
)

tensor_zero_ids = set(
    enriched_lsa.loc[
        tensor_zero_mask.cpu().numpy(),
        "book_id"
    ]
)

print(
    "\nZero-vector book IDs match exactly:",
    source_zero_ids == tensor_zero_ids
)

print(
    "Zero-vector IDs:",
    sorted(tensor_zero_ids)
)


# ------------------------------------------------------------
# Validate non-zero vectors
# ------------------------------------------------------------

nonzero_norms = tensor_norms[
    ~tensor_zero_mask
]

print(
    "\nNon-zero vectors:",
    len(nonzero_norms)
)

print(
    "Minimum non-zero norm:",
    nonzero_norms.min().item()
)

print(
    "Maximum non-zero norm:",
    nonzero_norms.max().item()
)

print(
    "All non-zero norms > 0:",
    bool(
        torch.all(
            nonzero_norms > 0
        ).item()
    )
)

ZERO-VECTOR INTEGRITY VALIDATION
Source zero-vector count: 27
Tensor zero-vector count: 27
Counts match: True

Zero-vector book IDs match exactly: True
Zero-vector IDs: ['BOOK00963', 'BOOK01002', 'BOOK01030', 'BOOK01062', 'BOOK01072', 'BOOK01170', 'BOOK01173', 'BOOK01180', 'BOOK01306', 'BOOK01399', 'BOOK01450', 'BOOK01561', 'BOOK01659', 'BOOK01720', 'BOOK01733', 'BOOK01751', 'BOOK01799', 'BOOK01830', 'BOOK01940', 'BOOK01947', 'BOOK01967', 'BOOK01988', 'BOOK01991', 'BOOK02009', 'BOOK02012', 'BOOK02021', 'BOOK02065']

Non-zero vectors: 2040
Minimum non-zero norm: 0.01616101898252964
Maximum non-zero norm: 0.9790317416191101
All non-zero norms > 0: True


## 8. Save PyTorch Tensor Artifacts

The validated PyTorch feature tensor is saved as a reusable model artifact.

Two artifacts are retained:

1. **Book feature tensor** — the `2,067 × 200` `float32` tensor containing the enriched LSA representation.
2. **Tensor book index** — the ordered mapping between tensor rows and `book_id`.

The book index is saved separately because PyTorch tensors contain numerical values only and do not retain book identifiers.

Together, these artifacts allow subsequent notebooks and applications to reload the tensor while preserving exact book-to-row alignment.

The saved tensor remains on the **CPU** for portability. It can be transferred to MPS, CUDA, or another supported device at runtime when required.

In [13]:
# ============================================================
# SAVE PYTORCH TENSOR ARTIFACTS
# ============================================================

TENSOR_PATH = (
    MODELS_DIR
    / "enriched_lsa_200_tensor.pt"
)

TENSOR_INDEX_PATH = (
    PROCESSED_DIR
    / "pytorch_tensor_book_index.csv"
)


# ------------------------------------------------------------
# Save canonical CPU tensor
# ------------------------------------------------------------

torch.save(
    book_feature_tensor,
    TENSOR_PATH
)


# ------------------------------------------------------------
# Save row-to-book mapping
# ------------------------------------------------------------

tensor_book_index = enriched_lsa[
    [
        "book_id",
        "is_zero_vector"
    ]
].copy()

tensor_book_index.insert(
    0,
    "tensor_row",
    np.arange(
        len(tensor_book_index)
    )
)

tensor_book_index.to_csv(
    TENSOR_INDEX_PATH,
    index=False
)


# ------------------------------------------------------------
# Reload tensor for validation
# ------------------------------------------------------------

reloaded_tensor = torch.load(
    TENSOR_PATH,
    map_location="cpu"
)

reloaded_index = pd.read_csv(
    TENSOR_INDEX_PATH
)


# ------------------------------------------------------------
# Validate saved artifacts
# ------------------------------------------------------------

print("PYTORCH ARTIFACT VALIDATION")
print("=" * 80)

print(
    "Tensor saved:",
    TENSOR_PATH.exists()
)

print(
    "Index saved:",
    TENSOR_INDEX_PATH.exists()
)

print(
    "\nReloaded tensor shape:",
    reloaded_tensor.shape
)

print(
    "Reloaded tensor dtype:",
    reloaded_tensor.dtype
)

print(
    "Reloaded tensor device:",
    reloaded_tensor.device
)

print(
    "\nTensor values preserved:",
    torch.equal(
        book_feature_tensor,
        reloaded_tensor
    )
)

print(
    "Index rows:",
    len(reloaded_index)
)

print(
    "Index aligned with tensor:",
    len(reloaded_index)
    == reloaded_tensor.shape[0]
)

print(
    "Book ID order preserved:",
    reloaded_index["book_id"].equals(
        enriched_lsa["book_id"]
        .reset_index(drop=True)
    )
)

print(
    "Zero-vector flags preserved:",
    (
        reloaded_index["is_zero_vector"]
        .astype(bool)
        .to_numpy()
        ==
        enriched_lsa["is_zero_vector"]
        .to_numpy()
    ).all()
)

print("\nSaved artifacts:")
print(TENSOR_PATH)
print(TENSOR_INDEX_PATH)

PYTORCH ARTIFACT VALIDATION
Tensor saved: True
Index saved: True

Reloaded tensor shape: torch.Size([2067, 200])
Reloaded tensor dtype: torch.float32
Reloaded tensor device: cpu

Tensor values preserved: True
Index rows: 2067
Index aligned with tensor: True
Book ID order preserved: True
Zero-vector flags preserved: True

Saved artifacts:
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/models/enriched_lsa_200_tensor.pt
/Users/jannoelvero/Documents/Ironhack/Week_11/Leadership_Management_Book_Recommendation_System/data/processed/pytorch_tensor_book_index.csv


# Notebook 13 — PyTorch Tensors: Final Summary

## Objective

This notebook prepared the numerical book representations developed in the previous stages for use with **PyTorch**.

The project remains primarily an **unsupervised, content-based recommendation system**. No artificial supervised target variable was introduced.

---

## Input Representation

The tensor representation was created from the validated:

**Enriched LSA 200-dimensional representation**

generated from the enriched TF-IDF feature space using Truncated SVD.

The enriched representation originates from:

**Title + Authors + Subjects + Description**

The original enriched TF-IDF matrix remains the primary representation used by the production recommendation engine for exact cosine-similarity ranking.

The LSA representation is used here because it provides a compact, dense numerical representation suitable for PyTorch.

---

## Tensor Structure

The final PyTorch feature tensor contains:

- **2,067 books**
- **200 latent semantic features per book**
- Shape: `torch.Size([2067, 200])`
- Data type: `torch.float32`
- Canonical storage device: `CPU`

Each tensor row corresponds to one book in the preserved book index.

No missing or infinite numerical values were detected.

---

## Tensor Operations

PyTorch operations were successfully demonstrated for:

- individual book-vector retrieval;
- batch selection;
- vector norms;
- cosine similarity;
- L2 normalization;
- device transfer.

A five-book test batch was successfully normalized to unit-length vectors.

Cosine similarity in this notebook operates on the reduced LSA representation and is used to demonstrate tensor functionality. It does not replace the enriched TF-IDF cosine-similarity calculation used by the primary recommendation system.

---

## Apple Silicon Acceleration

PyTorch detected the Apple Metal Performance Shaders backend:

**MPS available: True**

The complete tensor was successfully transferred from CPU to `mps:0`, and numerical operations were successfully executed on the accelerated device.

The canonical saved tensor remains on CPU for portability and can be transferred to an available accelerator at runtime.

---

## Dataset and DataLoader

A custom PyTorch `Dataset` was created to maintain the relationship between:

**book_id → 200-dimensional feature vector**

A PyTorch `DataLoader` was then configured with:

- Dataset size: **2,067 books**
- Batch size: **32**
- Number of batches: **65**
- Feature batch shape: `[32, 200]`

Book identifiers and tensor vectors remained correctly aligned through batch loading.

No target variable was included because the current task is not formulated as a supervised prediction problem.

---

## Zero-Vector Integrity

The enriched NLP representation previously contained **27 zero-vector books**.

Tensor validation confirmed:

- Source zero vectors: **27**
- Tensor zero vectors: **27**
- Exact zero-vector book identities preserved: **Yes**
- Non-zero vectors: **2,040**

The non-zero tensor vectors had norms ranging from approximately:

- Minimum: **0.0162**
- Maximum: **0.9790**

No artificial imputation was introduced for books without usable semantic vectors.

---

## Saved Artifacts

Two reusable artifacts were created:

### PyTorch Feature Tensor

`models/enriched_lsa_200_tensor.pt`

Contains the validated:

**2,067 × 200 float32 PyTorch tensor**

### Tensor Book Index

`data/processed/pytorch_tensor_book_index.csv`

Contains the ordered mapping between:

- tensor row;
- `book_id`;
- zero-vector status.

---

## Artifact Validation

The saved artifacts were reloaded successfully.

Validation confirmed:

- Tensor file exists: **True**
- Index file exists: **True**
- Tensor values preserved exactly: **True**
- Tensor/index row alignment: **True**
- Book ID ordering preserved: **True**
- Zero-vector flags preserved: **True**

---

## Methodological Position

PyTorch is introduced as a numerical and deep-learning framework without changing the underlying formulation of the project.

The validated recommendation architecture remains:

**Enriched TF-IDF → Cosine Similarity → Duplicate Suppression → Ranked Top-N Recommendations**

The PyTorch tensor provides a model-ready dense representation that can support subsequent neural-network experimentation where methodologically appropriate.

---

## Final Status

**Notebook 13 — PyTorch Tensors: COMPLETE**

Final tensor pipeline:

**Enriched TF-IDF → Truncated SVD / LSA (200 dimensions) → NumPy float32 → PyTorch Tensor → Dataset / DataLoader → Validated Portable Artifact**

The tensor representation is ready for the next modelling stage.